In [ ]:
import glob
import pandas as pd

OUT_PATH = "submission.csv"
RAUNAK_TOKENS = ["aleixlopez", "birdclef-2026-ported-raunak-v6-v1"]
MEENAL_TOKENS = ["aleixlopez", "birdclef-2026-meenal-v1"]
EOS6_TOKENS = ["birdclef-2026-eos-9"]
WEIGHTS = {
    "raunak": 0.35,
    "meenal": 0.35,
    "eos6": 0.30,
}
SOUNDSCAPE_ONLY_CLASSES = {"1491113", "25073", "47158son01", "47158son02", "47158son03", "47158son04", "47158son05", "47158son06", "47158son07", "47158son08", "47158son09", "47158son10", "47158son11", "47158son12", "47158son13", "47158son14", "47158son15", "47158son16", "47158son17", "47158son18", "47158son19", "47158son20", "47158son21", "47158son22", "47158son23", "47158son24", "47158son25", "517063"}
SOUNDSCAPE_BOOST_FACTOR = 1.08
USE_NON_BIRD_FACTORS = True
NON_BIRD_FACTOR_MAP = {
    "1161364": 1.10,
    "116570": 1.20,
    "1176823": 1.08,
    "1491113": 1.08,
    "1595929": 1.08,
    "209233": 1.06,
    "22930": 1.08,
    "22956": 1.08,
    "22961": 1.08,
    "22967": 1.08,
    "22973": 1.08,
    "22983": 1.08,
    "22985": 1.08,
    "23150": 1.08,
    "23154": 1.08,
    "23158": 1.08,
    "23176": 1.08,
    "23724": 1.08,
    "24279": 1.08,
    "24285": 1.08,
    "24287": 1.08,
    "24321": 1.08,
    "244024": 1.10,
    "25073": 1.08,
    "25092": 1.08,
    "25214": 1.08,
    "326272": 1.08,
    "41970": 1.06,
    "43435": 1.06,
    "47144": 1.06,
    "47158son01": 1.10,
    "47158son02": 1.10,
    "47158son03": 1.10,
    "47158son04": 1.10,
    "47158son05": 1.10,
    "47158son06": 1.10,
    "47158son07": 1.10,
    "47158son08": 1.10,
    "47158son09": 1.10,
    "47158son10": 1.10,
    "47158son11": 1.10,
    "47158son12": 1.10,
    "47158son13": 1.10,
    "47158son14": 1.10,
    "47158son15": 1.10,
    "47158son16": 1.10,
    "47158son17": 1.10,
    "47158son18": 1.10,
    "47158son19": 1.10,
    "47158son20": 1.10,
    "47158son21": 1.10,
    "47158son22": 1.10,
    "47158son23": 1.10,
    "47158son24": 1.10,
    "47158son25": 1.10,
    "476521": 1.08,
    "516975": 1.06,
    "517063": 1.08,
    "555123": 1.08,
    "555145": 1.08,
    "555146": 1.08,
    "64898": 1.08,
    "65377": 1.08,
    "65380": 1.08,
    "66971": 1.08,
    "67107": 1.08,
    "67252": 1.08,
    "70711": 1.08,
    "738183": 1.06,
    "74113": 1.06,
    "74580": 1.06,
    "760266": 1.10
}


def _find_submission(required_tokens):
    candidates = glob.glob("/kaggle/input/notebooks/**/submission.csv", recursive=True)
    matches = []
    for path in candidates:
        normalized = path.lower().replace("\\", "/")
        if all(token in normalized for token in required_tokens):
            matches.append(path)
    if not matches:
        raise FileNotFoundError(f"Could not find submission.csv matching tokens={required_tokens}")
    return sorted(matches)[-1]


def _find_sample_submission():
    candidates = glob.glob("/kaggle/input/**/sample_submission.csv", recursive=True)
    if not candidates:
        raise FileNotFoundError("sample_submission.csv not found under /kaggle/input")
    return sorted(candidates)[0]


def _align_to_template(df, row_ids, class_cols, source_name):
    if "row_id" not in df.columns:
        raise ValueError(f"{source_name}: source submission is missing row_id")
    missing_cols = [c for c in class_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"{source_name}: source submission missing class columns: {missing_cols[:5]}")
    source = df[["row_id"] + class_cols].copy()
    source["row_id"] = source["row_id"].astype(str).str.strip()
    source = source.drop_duplicates(subset=["row_id"]).set_index("row_id")
    template_index = pd.Index(row_ids)
    missing_rows = int((~template_index.isin(source.index)).sum())
    if missing_rows:
        raise ValueError(
            f"{source_name}: {missing_rows} row_id values from sample_submission are missing in source submission"
        )
    aligned = source.reindex(row_ids)
    if aligned.isna().any().any():
        raise ValueError(f"{source_name}: NaNs found after row_id alignment; refusing to generate unsafe blend")
    return aligned


raunak_path = _find_submission(RAUNAK_TOKENS)
meenal_path = _find_submission(MEENAL_TOKENS)
eos6_path = _find_submission(EOS6_TOKENS)
sample_path = _find_sample_submission()
print("Using Raunak submission:", raunak_path)
print("Using Meenal submission:", meenal_path)
print("Using EoS6 submission:", eos6_path)
print("Using sample submission template:", sample_path)

raunak = pd.read_csv(raunak_path)
meenal = pd.read_csv(meenal_path)
eos6 = pd.read_csv(eos6_path)
sample = pd.read_csv(sample_path)

if "row_id" not in sample.columns:
    raise ValueError("sample_submission.csv missing row_id")
sample["row_id"] = sample["row_id"].astype(str).str.strip()
class_cols = [c for c in sample.columns if c != "row_id"]
row_ids = sample["row_id"].tolist()

raunak_aligned = _align_to_template(raunak, row_ids, class_cols, "Raunak")
meenal_aligned = _align_to_template(meenal, row_ids, class_cols, "Meenal")
eos6_aligned = _align_to_template(eos6, row_ids, class_cols, "EoS6")

merged = sample[["row_id"]].copy()
for c in class_cols:
    rank_raunak = raunak_aligned[c].rank(method="average", pct=True)
    rank_meenal = meenal_aligned[c].rank(method="average", pct=True)
    rank_eos6 = eos6_aligned[c].rank(method="average", pct=True)
    merged[c] = (
        WEIGHTS["raunak"] * rank_raunak
        + WEIGHTS["meenal"] * rank_meenal
        + WEIGHTS["eos6"] * rank_eos6
    )

if SOUNDSCAPE_BOOST_FACTOR > 1.0:
    for c in SOUNDSCAPE_ONLY_CLASSES:
        if c in merged.columns:
            merged[c] = (merged[c] * SOUNDSCAPE_BOOST_FACTOR).clip(0.0, 1.0)

if USE_NON_BIRD_FACTORS:
    for c, factor in NON_BIRD_FACTOR_MAP.items():
        if c in merged.columns:
            merged[c] = (merged[c] * factor).clip(0.0, 1.0)

merged = merged[["row_id"] + class_cols]
if merged[class_cols].isna().any().any():
    raise ValueError("Blend produced NaNs; aborting submission generation")
merged.to_csv(OUT_PATH, index=False, float_format="%.10f")
print(merged.shape)
print(f"Wrote {OUT_PATH}")
